<a href="https://colab.research.google.com/github/sergo8online/CIFAR-10-with-MLP/blob/main/CIFAR_with_MLP_Github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torchmetrics
import torch
from torch import nn
import matplotlib.pyplot as plt
import torchvision
from torchvision import transforms
from torchvision.transforms import ToTensor, Normalize, RandomHorizontalFlip, Resize
from torchvision import datasets
from torchmetrics import Accuracy

print(torch.__version__)
print(torchvision.__version__)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
torch.manual_seed(42)
train_transform = transforms.Compose([Resize((64, 64)),
                                RandomHorizontalFlip(),
                                ToTensor(),
                                Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
test_transform = transforms.Compose([Resize((64, 64)),
                                    ToTensor(),
                                    Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [ ]:
torch.manual_seed(42)
train_data = datasets.CIFAR10(root="data",
                              train=True,
                              transform=train_transform,
                              target_transform=None,
                              download=True)
test_data = datasets.CIFAR10(root='data',
                             train=False,
                             transform=test_transform,
                             target_transform=None,
                             download=True)

In [ ]:
len(train_data), len(test_data)

In [ ]:
classes = train_data.classes
classes

In [ ]:
train_data.class_to_idx

In [ ]:
image, label = train_data[0]
image.size()                         #so it is channel first, ok, and also 32*32

In [ ]:
import random
fig = plt.figure(figsize=(9, 9))
rows, cols = 4, 4
for i in range(1, rows*cols+1):
  idx = random.randint(0, len(train_data))
  image, label = train_data[idx]
  image = torch.permute(image, (1, 2, 0))
  fig.add_subplot(rows, cols, i)
  plt.title(classes[label])
  plt.imshow(image)
  plt.axis(False)

In [ ]:
idx = random.randint(0, len(train_data))
image, label = train_data[idx]
flatten = nn.Flatten()
linear = nn.Linear(in_features = 4096,
                   out_features = len(classes))
flatten(image).shape
A = linear(flatten(image))
print(A)
print()
y_logit = torch.softmax(A, dim=1).argmax(dim=1)
y_logit

In [ ]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(dataset=train_data,
                              batch_size=32,
                              shuffle=True)
test_dataloader = DataLoader(dataset = test_data,
                             batch_size=32,
                             shuffle=False)

In [ ]:
class MLP(nn.Module):
  def __init__(self, input_shape, output_shape):
    super().__init__()
    self.layer_stack = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=input_shape,
                  out_features = 2048),
        nn.BatchNorm1d(2048),
        nn.LeakyReLU(),
        nn.Dropout(0.3),
        nn.Linear(in_features=2048,
                  out_features = 512),
        nn.BatchNorm1d(512),
        nn.LeakyReLU(),
        nn.Dropout(0.3),
        nn.Linear(in_features = 512,
                  out_features=128),
        nn.BatchNorm1d(128),
        nn.LeakyReLU(),
        nn.Dropout(0.3),
        nn.Linear(in_features=128,
                  out_features = 32),
        nn.BatchNorm1d(32),
        nn.LeakyReLU(),
        nn.Dropout(0.3),
        nn.Linear(in_features = 32,
                  out_features = output_shape)
    )
  def forward(self, x):
    return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)
model_mlp = MLP(input_shape = 3*64*64,
                output_shape = len(classes)).to(device)
model_mlp

In [ ]:
loss_fn = nn.CrossEntropyLoss()
acc_fn = Accuracy(task='multiclass', num_classes = len(classes))
optimizer = torch.optim.Adam(params = model_mlp.parameters())

In [ ]:
X_dummy = torch.rand(32, 3, 64, 64).to(device)
y_logits = model_mlp(X_dummy)
y_proba = torch.softmax(y_logits, dim=1)
y_preds = y_proba.argmax(dim=1)
print(y_logits[:1, :])
print(y_proba[:1, :])
print(y_preds[0])

In [ ]:
lambda_reg = 0.0001
def train_step(model: nn.Module,
               data_loader: DataLoader,
               loss_fn: nn.Module,
               acc_fn,
               optimizer: torch.optim,
               device: torch.device):
  model = model.to(device)
  model.train()
  acc_fn = acc_fn.to(device)

  train_loss, train_acc = 0, 0

  for X, y in data_loader:
    X, y = X.to(device), y.to(device)

    y_logits = model(X)
    y_preds = torch.softmax(y_logits, dim=1).argmax(dim=1)

    loss = loss_fn(y_logits, y)
    acc = acc_fn(y_preds, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_loss += loss
    train_acc += acc

  train_loss /= len(data_loader)
  train_acc /= len(data_loader)

  print(f"Train Loss: {train_loss} | Train Acc: {train_acc}")

In [ ]:
def test_step(model: nn.Module,
              data_loader: DataLoader,
              loss_fn: nn.Module,
              acc_fn,
              device: torch.device):
  model = model.to(device)
  model.eval()
  acc_fn = acc_fn.to(device)
  test_loss, test_acc = 0, 0

  with torch.inference_mode():
    for X, y in data_loader:
      X, y = X.to(device), y.to(device)

      y_logits = model(X)
      y_preds = torch.softmax(y_logits, dim=1).argmax(dim=1)

      test_loss += loss_fn(y_logits, y)
      test_acc += acc_fn(y_preds, y)

    test_loss /= len(data_loader)
    test_acc /= len(data_loader)

  print(f"Test Loss: {test_loss} | Test Acc: {test_acc}")

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
from tqdm.auto import tqdm
epochs = 100
for epoch in tqdm(range(epochs)):
  print(f"Epoch ========> {epoch}")
  train_step(model = model_mlp,
             data_loader=train_dataloader,
             loss_fn = loss_fn,
             acc_fn = acc_fn,
             optimizer = optimizer,
             device = device)
  test_step(model = model_mlp,
            data_loader = test_dataloader,
            loss_fn = loss_fn,
            acc_fn = acc_fn,
            device = device)

### Մեկնաբանություն

այստեղ ես կնկարագրեմ թե ինչեր եմ արել ու որը ինչ արդյունք է տվել։
քայլերը դասավորված են ժամանակագրական հաջորդականությամբ։
Սկզբնական պահին

--->
*   model = {Flatten,
            Linear(3072, 512),
            Linear(512, 128),
            Linear(128, 32),
            Linear(32, 10)}
*   optimizer = SGD(lr=0.1)
*   epochs = 10

**Արդյունքներ**


*   test accuracy = 0.33

**Խնդիրներ**


*   Թռչկոտող գրադիենտներ, որի հետևանքով թեստի accuracy-ն չի աճում։

**Լուծում**


*   lr = 0.01
*   momentum = 0.2

**Հետևանք**


*   train accuracy = 0.4
*   test accuracy = 0.39

**Խնդիր**


*   չկա նորմալիզացիյա


**Լուծում**


*   Օգտագործ ենք torchvision.transforms.Normalize
*   Օգտագործ ենք նաև nn.BatchNorm1d


**Արդյունք**


*   Միայն տվյալների նորմալիզացիաով --> train = 0.68, test = 0.53
*   Երկուսը իրար հետ ----------------> train = 0.64, test = 0.55


**Խնդիր**


*   Քանի որ train և test accuracy ները ահագին տարբերվում են կա մեծ ռիսկ overfit-ի

**Լուծում**


*   ավելացնենք Dropout 0.3-ով կամ 0.1-ով
*   անել աուգմենտացիա Resize((64, 64)) և RandomHorizontalFlip()

**Արդյունքներ**

*   Dropout(0.1)-ը 10 epoch-ի վրա տալիս է train = 0.58, test = 0.56, բայց լավ չի ցույց տալիս իրան 100 epoch-ի դեպքում -> train = 0.66, test = 0.56։ Իսկ Dropout(0.3)-ը հակառակը, դրա համար ես ընտրել եմ 0․3-ը։  
*   սրան ավելացրած աուգմենտացիան ստացվում է հետևյալը։
10 epoch ---> train = 0.54, test = 0.56
20 epoch ---> train = 0.59, test = 0.59
100 epoch --> train = 0.71, test = 0.63